# Bank marketing model - FINAL v2 (use this one, NOT v1!!)

Predicts if a client will subscribe to a term deposit after a sales call.

Last run on Kasun's laptop. **Don't change anything below, it works.**

In [ ]:
import os, sys, pickle, zipfile, io, urllib.request
import warnings
warnings.filterwarnings('ignore')   # too many warnings
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

In [ ]:
# download the data if we don't have it (copied from stackoverflow)
if not os.path.exists('bank-additional-full.csv'):
    z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(
        'https://archive.ics.uci.edu/static/public/222/bank+marketing.zip').read()))
    inner = zipfile.ZipFile(io.BytesIO(z.read('bank-additional.zip')))
    open('bank-additional-full.csv', 'wb').write(
        inner.read('bank-additional/bank-additional-full.csv'))
df = pd.read_csv('bank-additional-full.csv', sep=';')
print(df.shape)
df.head()

In [ ]:
df['y'] = df['y'].map({'yes': 1, 'no': 0})
df = pd.get_dummies(df, drop_first=True)   # encode everything
X = df.drop(columns=['y'])
y = df['y']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
model = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1)  # tuned by hand on friday
model.fit(X_train, y_train)

In [ ]:
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]
print('Accuracy:', round(accuracy_score(y_test, pred), 3))
print('ROC AUC: ', round(roc_auc_score(y_test, prob), 3))
# amazing!!! ship it

In [ ]:
# which features matter most? (how much AUC drops when a column is shuffled)
from sklearn.inspection import permutation_importance
r = permutation_importance(model, X_test, y_test, scoring='roc_auc', n_repeats=3, random_state=0)
imp = pd.Series(r.importances_mean, index=X.columns).sort_values().tail(10)
imp.plot.barh(figsize=(7, 4), title='Top 10 features')
plt.tight_layout()
plt.show()

In [ ]:
pickle.dump(model, open('model_final_v2_REAL.pkl', 'wb'))
print('saved model_final_v2_REAL.pkl - emailed to the engineering team')

In [ ]:
X2_train = X_train.drop(columns=['duration'])
X2_test = X_test.drop(columns=['duration'])
model2 = XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1).fit(X2_train, y_train)
print('ROC AUC with duration:   ', round(roc_auc_score(y_test, prob), 3))
print('ROC AUC without duration:', round(roc_auc_score(y_test, model2.predict_proba(X2_test)[:, 1]), 3))